In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


##_______________________________________________________________ "" ____________________________________________________________

##_______________________________________________________________ "" ____________________________________________________________

In [5]:
# ============================================================
# DEEP DIVE — WHATSAPP PERFORMANCE IN DPD 45-60
#
# QUESTION:
# After surviving until DPD 45-60, does another WhatsApp
# attempt still show economic value?
#
# Grain: 1 row = 1 historical WhatsApp send
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE DATA
# ============================================================

wa_4560 = wa.copy()

wa_4560["sent_at"] = pd.to_datetime(
    wa_4560["sent_at"]
)

wa_4560["days_past_due"] = pd.to_numeric(
    wa_4560["days_past_due"],
    errors="coerce"
)

wa_4560["amount_paid_brl"] = pd.to_numeric(
    wa_4560["amount_paid_brl"],
    errors="coerce"
).fillna(0)

wa_4560["outstanding_balance_brl"] = pd.to_numeric(
    wa_4560["outstanding_balance_brl"],
    errors="coerce"
)


# ============================================================
# 2. KEEP ONLY DPD 45-60
# ============================================================

wa_4560 = (
    wa_4560.loc[
        wa_4560["days_past_due"].between(
            45,
            60
        )
    ]
    .copy()
)


# ============================================================
# 3. BASIC RESPONSE VARIABLES
# ============================================================

wa_4560["paid_72h"] = (
    wa_4560["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

wa_4560["payment_amount_72h"] = (
    wa_4560["amount_paid_brl"]
)

wa_4560["recovery_per_message"] = (
    wa_4560["payment_amount_72h"]
)


# ============================================================
# 4. HIGH-LEVEL PERFORMANCE
# ============================================================

n_messages = len(wa_4560)

n_customers = (
    wa_4560["customer_id"]
    .nunique()
)

n_payers = (
    wa_4560.loc[
        wa_4560["paid_72h"],
        "customer_id"
    ]
    .nunique()
)

total_recovery = (
    wa_4560["payment_amount_72h"]
    .sum()
)


print("=" * 110)
print("HISTORICAL WHATSAPP — DPD 45-60")
print("=" * 110)

print(
    f"Messages                     : "
    f"{n_messages:,}"
)

print(
    f"Unique customers             : "
    f"{n_customers:,}"
)

print(
    f"Customers paying within 72h  : "
    f"{n_payers:,}"
)

print(
    f"Payment events within 72h    : "
    f"{wa_4560['paid_72h'].sum():,}"
)

print(
    f"Payment response rate        : "
    f"{wa_4560['paid_72h'].mean():.2%}"
)

print(
    f"Observed recovery            : "
    f"R$ {total_recovery:,.2f}"
)

print(
    f"Recovery / message           : "
    f"R$ {total_recovery / n_messages:,.2f}"
)

print(
    f"WhatsApp cost                : "
    f"R$ {n_messages:,.2f}"
)

print(
    f"Observed recovery - WA cost  : "
    f"R$ {total_recovery - n_messages:,.2f}"
)


# ============================================================
# 5. PERFORMANCE BY EXACT DPD
# ============================================================

by_exact_dpd = (
    wa_4560
    .groupby(
        "days_past_due"
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


by_exact_dpd["recovery_per_message"] = (
    by_exact_dpd["recovery_brl"]
    /
    by_exact_dpd["messages"]
)

by_exact_dpd["cost_brl"] = (
    by_exact_dpd["messages"]
)

by_exact_dpd["observed_net_brl"] = (
    by_exact_dpd["recovery_brl"]
    -
    by_exact_dpd["cost_brl"]
)


print("\n" + "=" * 110)
print("PERFORMANCE BY EXACT DPD")
print("=" * 110)

display(
    by_exact_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}",
        "cost_brl": "R$ {:,.2f}",
        "observed_net_brl": "R$ {:,.2f}"
    })
)


# ============================================================
# 6. SUB-BUCKETS INSIDE 45-60
#
# We want to know whether the signal decays as we approach 60.
# ============================================================

wa_4560["dpd_subbucket"] = pd.cut(
    wa_4560["days_past_due"],
    bins=[
        44,
        49,
        54,
        60
    ],
    labels=[
        "45-49",
        "50-54",
        "55-60"
    ]
)


by_subbucket = (
    wa_4560
    .groupby(
        "dpd_subbucket",
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


by_subbucket["recovery_per_message"] = (
    by_subbucket["recovery_brl"]
    /
    by_subbucket["messages"]
)


print("\n" + "=" * 110)
print("PERFORMANCE WITHIN DPD 45-60")
print("=" * 110)

display(
    by_subbucket.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}"
    })
)


# ============================================================
# 7. PRESSURE — NUMBER OF MESSAGES IN LAST 14 DAYS
# ============================================================

wa_4560["pressure_bucket"] = pd.cut(
    wa_4560["n_msgs_last_14d"],
    bins=[
        -1,
        0,
        2,
        4,
        6,
        np.inf
    ],
    labels=[
        "0",
        "1-2",
        "3-4",
        "5-6",
        "7+"
    ]
)


by_pressure = (
    wa_4560
    .groupby(
        "pressure_bucket",
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_dpd=(
            "days_past_due",
            "mean"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


by_pressure["recovery_per_message"] = (
    by_pressure["recovery_brl"]
    /
    by_pressure["messages"]
)


print("\n" + "=" * 110)
print("PERFORMANCE BY CONTACT PRESSURE — DPD 45-60")
print("=" * 110)

display(
    by_pressure.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_dpd": "{:.1f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# 8. DPD × PRESSURE
#
# Critical view:
# Does timing interact with previous contact saturation?
# ============================================================

dpd_pressure = (
    wa_4560
    .groupby(
        [
            "dpd_subbucket",
            "pressure_bucket"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        )
    )
    .reset_index()
)


dpd_pressure["recovery_per_message"] = (
    dpd_pressure["recovery_brl"]
    /
    dpd_pressure["messages"]
)


print("\n" + "=" * 110)
print("DPD × CONTACT PRESSURE")
print("=" * 110)

display(
    dpd_pressure.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)


# ============================================================
# 9. TEMPLATE × DPD
# ============================================================

template_dpd = (
    wa_4560
    .groupby(
        [
            "dpd_subbucket",
            "template"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


template_dpd["recovery_per_message"] = (
    template_dpd["recovery_brl"]
    /
    template_dpd["messages"]
)


print("\n" + "=" * 110)
print("TEMPLATE × DPD — 45-60")
print("=" * 110)

display(
    template_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}"
    })
)


# ============================================================
# 10. INTERACTION × DPD
# ============================================================

interaction_dpd = (
    wa_4560
    .groupby(
        [
            "dpd_subbucket",
            "interaction"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        )
    )
    .reset_index()
)


interaction_dpd["recovery_per_message"] = (
    interaction_dpd["recovery_brl"]
    /
    interaction_dpd["messages"]
)


print("\n" + "=" * 110)
print("INTERACTION × DPD — 45-60")
print("=" * 110)

display(
    interaction_dpd.style.format({
        "messages": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)

HISTORICAL WHATSAPP — DPD 45-60
Messages                     : 5,381
Unique customers             : 3,019
Customers paying within 72h  : 258
Payment events within 72h    : 260
Payment response rate        : 4.83%
Observed recovery            : R$ 139,382.61
Recovery / message           : R$ 25.90
WhatsApp cost                : R$ 5,381.00
Observed recovery - WA cost  : R$ 134,001.61

PERFORMANCE BY EXACT DPD


,days_past_due,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,avg_prior_msgs,recovery_per_message,cost_brl,observed_net_brl
0,45,418,418,31,7.42%,"R$ 14,129.30",R$ 790.92,1.38,R$ 33.80,R$ 418.00,"R$ 13,711.30"
1,46,395,395,14,3.54%,"R$ 9,497.03",R$ 758.25,1.41,R$ 24.04,R$ 395.00,"R$ 9,102.03"
2,47,380,380,13,3.42%,"R$ 7,509.01",R$ 805.47,1.46,R$ 19.76,R$ 380.00,"R$ 7,129.01"
3,48,402,402,20,4.98%,"R$ 12,072.85",R$ 797.97,1.41,R$ 30.03,R$ 402.00,"R$ 11,670.85"
4,49,343,343,12,3.50%,"R$ 4,979.07",R$ 788.15,1.36,R$ 14.52,R$ 343.00,"R$ 4,636.07"
5,50,391,391,21,5.37%,"R$ 11,195.09",R$ 797.25,1.34,R$ 28.63,R$ 391.00,"R$ 10,804.09"
6,51,405,405,25,6.17%,"R$ 11,568.02",R$ 794.45,1.39,R$ 28.56,R$ 405.00,"R$ 11,163.02"
7,52,331,331,16,4.83%,"R$ 5,561.86",R$ 786.68,1.29,R$ 16.80,R$ 331.00,"R$ 5,230.86"
8,53,327,327,15,4.59%,"R$ 9,730.87",R$ 794.39,1.42,R$ 29.76,R$ 327.00,"R$ 9,403.87"
9,54,320,320,12,3.75%,"R$ 5,872.05",R$ 792.82,1.41,R$ 18.35,R$ 320.00,"R$ 5,552.05"



PERFORMANCE WITHIN DPD 45-60


,dpd_subbucket,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,avg_prior_msgs,recovery_per_message
0,45-49,"1,938","1,634",90,4.64%,"R$ 48,187.26",R$ 788.09,1.40,R$ 24.86
1,50-54,"1,774","1,471",89,5.02%,"R$ 43,927.89",R$ 793.31,1.37,R$ 24.76
2,55-60,"1,669","1,351",81,4.85%,"R$ 47,267.46",R$ 778.08,1.30,R$ 28.32



PERFORMANCE BY CONTACT PRESSURE — DPD 45-60


...

,dpd_subbucket,pressure_bucket,messages,customers,payment_rate,recovery_brl,recovery_per_message
0,45-49,0,424,424,6.84%,"R$ 19,857.07",R$ 46.83
1,45-49,1-2,"1,226","1,084",4.40%,"R$ 23,504.94",R$ 19.17
2,45-49,3-4,270,235,2.59%,"R$ 4,825.25",R$ 17.87
3,45-49,5-6,18,16,0.00%,R$ 0.00,R$ 0.00
4,50-54,0,408,408,6.13%,"R$ 10,000.89",R$ 24.51
5,50-54,1-2,"1,100",957,5.09%,"R$ 29,012.82",R$ 26.38
6,50-54,3-4,254,221,3.15%,"R$ 4,914.18",R$ 19.35
7,50-54,5-6,11,10,0.00%,R$ 0.00,R$ 0.00
8,50-54,7+,1,1,0.00%,R$ 0.00,R$ 0.00
9,55-60,0,431,431,4.64%,"R$ 8,498.36",R$ 19.72



TEMPLATE × DPD — 45-60


,dpd_subbucket,template,messages,customers,payment_events,payment_rate,recovery_brl,avg_prior_msgs,recovery_per_message
0,45-49,discount_offer,966,886,50,5.18%,"R$ 23,486.00",1.40,R$ 24.31
1,45-49,pix_link,385,369,19,4.94%,"R$ 8,808.22",1.43,R$ 22.88
2,45-49,urgent_reminder,587,552,21,3.58%,"R$ 15,893.04",1.39,R$ 27.08
3,50-54,discount_offer,876,801,57,6.51%,"R$ 23,426.93",1.39,R$ 26.74
4,50-54,pix_link,364,349,15,4.12%,"R$ 9,041.62",1.37,R$ 24.84
5,50-54,urgent_reminder,534,507,17,3.18%,"R$ 11,459.34",1.34,R$ 21.46
6,55-60,discount_offer,847,773,47,5.55%,"R$ 24,658.11",1.30,R$ 29.11
7,55-60,pix_link,356,341,12,3.37%,"R$ 8,458.64",1.20,R$ 23.76
8,55-60,urgent_reminder,466,437,22,4.72%,"R$ 14,150.71",1.40,R$ 30.37



INTERACTION × DPD — 45-60


,dpd_subbucket,interaction,messages,payment_rate,recovery_brl,recovery_per_message
0,45-49,clicked_link,214,14.02%,"R$ 20,673.97",R$ 96.61
1,45-49,none,"1,290",2.79%,"R$ 15,713.14",R$ 12.18
2,45-49,read,343,4.37%,"R$ 5,309.63",R$ 15.48
3,45-49,replied,91,9.89%,"R$ 6,490.52",R$ 71.32
4,50-54,clicked_link,196,13.78%,"R$ 12,146.37",R$ 61.97
5,50-54,none,"1,185",2.62%,"R$ 16,425.16",R$ 13.86
6,50-54,read,306,8.17%,"R$ 11,330.81",R$ 37.03
7,50-54,replied,87,6.90%,"R$ 4,025.55",R$ 46.27
8,55-60,clicked_link,183,15.30%,"R$ 16,575.72",R$ 90.58
9,55-60,none,"1,151",2.61%,"R$ 18,590.08",R$ 16.15


In [6]:
# ============================================================
# DPD 45-60 — "IS ONE MORE MESSAGE WORTH IT?"
#
# Grain:
# 1 row = 1 historical WhatsApp send at DPD 45-60
#
# IMPORTANT:
# All history variables below are calculated STRICTLY
# BEFORE the current message.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE FULL WHATSAPP HISTORY
# ============================================================

hist = wa.copy()

hist["sent_at"] = pd.to_datetime(
    hist["sent_at"]
)

hist["days_past_due"] = pd.to_numeric(
    hist["days_past_due"],
    errors="coerce"
)

hist["amount_paid_brl"] = pd.to_numeric(
    hist["amount_paid_brl"],
    errors="coerce"
).fillna(0)

hist["outstanding_balance_brl"] = pd.to_numeric(
    hist["outstanding_balance_brl"],
    errors="coerce"
)

hist = (
    hist
    .sort_values(
        ["customer_id", "sent_at"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. MESSAGE NUMBER IN CUSTOMER JOURNEY
#
# cumcount = number of messages STRICTLY BEFORE current one
# ============================================================

hist["n_prior_msgs_total"] = (
    hist
    .groupby("customer_id")
    .cumcount()
)


# ============================================================
# 3. PREVIOUS MESSAGE
# ============================================================

hist["prev_sent_at"] = (
    hist
    .groupby("customer_id")["sent_at"]
    .shift(1)
)

hist["days_since_prev_msg"] = (
    hist["sent_at"]
    -
    hist["prev_sent_at"]
).dt.total_seconds() / 86400


# ============================================================
# 4. PREVIOUS TEMPLATE / DELIVERY / INTERACTION
# ============================================================

hist["prev_template"] = (
    hist
    .groupby("customer_id")["template"]
    .shift(1)
)

hist["prev_delivery_status"] = (
    hist
    .groupby("customer_id")["delivery_status"]
    .shift(1)
)

hist["prev_interaction"] = (
    hist
    .groupby("customer_id")["interaction"]
    .shift(1)
)


# ============================================================
# 5. PRIOR ENGAGEMENT
#
# IMPORTANT:
# interaction from CURRENT message cannot be used.
#
# We only use interactions from PREVIOUS messages.
# ============================================================

hist["engaged_current"] = (
    hist["interaction"]
    .isin([
        "read",
        "clicked_link",
        "replied"
    ])
    .astype(int)
)

hist["clicked_current"] = (
    hist["interaction"]
    .eq("clicked_link")
    .astype(int)
)

hist["replied_current"] = (
    hist["interaction"]
    .eq("replied")
    .astype(int)
)


hist["had_prior_engagement"] = (
    hist
    .groupby("customer_id")["engaged_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["had_prior_click"] = (
    hist
    .groupby("customer_id")["clicked_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["had_prior_reply"] = (
    hist
    .groupby("customer_id")["replied_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


# ============================================================
# 6. PRIOR PAYMENT
#
# Again: only payments BEFORE current message.
# ============================================================

hist["payment_current"] = (
    hist["amount_paid_brl"] > 0
).astype(int)


hist["had_prior_payment"] = (
    hist
    .groupby("customer_id")["payment_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["prior_amount_paid_brl"] = (
    hist
    .groupby("customer_id")["amount_paid_brl"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)


# ============================================================
# 7. PRIOR CONTACT PRESSURE — 14 DAYS
#
# We already have n_msgs_last_14d in the original dataset.
#
# First verify that it is a pre-treatment feature.
# ============================================================

hist["n_msgs_last_14d"] = pd.to_numeric(
    hist["n_msgs_last_14d"],
    errors="coerce"
)


# ============================================================
# 8. KEEP CURRENT SENDS AT DPD 45-60
# ============================================================

decision = (
    hist.loc[
        hist["days_past_due"]
        .between(45, 60)
    ]
    .copy()
)


# ============================================================
# 9. OUTCOME OF CURRENT MESSAGE
# ============================================================

decision["paid_72h"] = (
    decision["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

decision["recovery_72h"] = (
    decision["amount_paid_brl"]
)


# ============================================================
# 10. DPD BUCKET
# ============================================================

decision["dpd_stage"] = pd.cut(
    decision["days_past_due"],
    bins=[
        44,
        49,
        54,
        60
    ],
    labels=[
        "45-49",
        "50-54",
        "55-60"
    ]
)


# ============================================================
# 11. TOTAL PRIOR MESSAGE BUCKET
#
# This is much more informative than only n_msgs_last_14d.
# ============================================================

decision["prior_msgs_bucket"] = pd.cut(
    decision["n_prior_msgs_total"],
    bins=[
        -1,
        2,
        5,
        8,
        np.inf
    ],
    labels=[
        "0-2",
        "3-5",
        "6-8",
        "9+"
    ]
)


# ============================================================
# 12. RECENCY BUCKET
# ============================================================

decision["recency_bucket"] = pd.cut(
    decision["days_since_prev_msg"],
    bins=[
        -np.inf,
        3,
        7,
        14,
        np.inf
    ],
    labels=[
        "0-3d",
        "4-7d",
        "8-14d",
        "15+d"
    ]
)


# ============================================================
# 13. PRIOR ENGAGEMENT PROFILE
#
# Strongest historical signal available BEFORE current send.
# ============================================================

decision["prior_engagement_profile"] = np.select(

    [
        decision["had_prior_click"],
        decision["had_prior_reply"],
        decision["had_prior_engagement"]
    ],

    [
        "prior_click",
        "prior_reply",
        "prior_read_only"
    ],

    default="no_prior_engagement"
)


# ============================================================
# 14. MAIN DECISION TABLE
#
# DPD × cumulative pressure × recency
# ============================================================

decision_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "prior_msgs_bucket",
            "recency_bucket"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        prior_engagement_rate=(
            "had_prior_engagement",
            "mean"
        ),

        prior_payment_rate=(
            "had_prior_payment",
            "mean"
        ),

        avg_msgs_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


decision_table["recovery_per_message"] = (
    decision_table["recovery_brl"]
    /
    decision_table["messages"]
)


decision_table["wa_cost"] = (
    decision_table["messages"]
)


decision_table["observed_net"] = (
    decision_table["recovery_brl"]
    -
    decision_table["wa_cost"]
)


# ============================================================
# 15. SAMPLE SIZE FLAG
# ============================================================

decision_table["sample_flag"] = np.select(

    [
        decision_table["messages"] >= 100,
        decision_table["messages"] >= 50,
        decision_table["messages"] >= 30
    ],

    [
        "GOOD",
        "OK",
        "SMALL"
    ],

    default="VERY_SMALL"
)


# ============================================================
# 16. SORT FOR BUSINESS READING
# ============================================================

decision_table = (
    decision_table
    .sort_values(
        [
            "dpd_stage",
            "prior_msgs_bucket",
            "recency_bucket"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 130)
print("ONE MORE MESSAGE? — DPD 45-60")
print("=" * 130)

display(
    decision_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "observed_net":
            "R$ {:,.2f}",

        "wa_cost":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "prior_engagement_rate":
            "{:.1%}",

        "prior_payment_rate":
            "{:.1%}",

        "avg_msgs_14d":
            "{:.2f}"
    })
)


# ============================================================
# 17. SECOND TABLE:
# PRIOR ENGAGEMENT
#
# This tells us whether previous signs of interest identify
# customers worth another attempt.
# ============================================================

engagement_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "prior_engagement_profile"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_days_since_prev_msg=(
            "days_since_prev_msg",
            "mean"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


engagement_table["recovery_per_message"] = (
    engagement_table["recovery_brl"]
    /
    engagement_table["messages"]
)


print("\n" + "=" * 130)
print("PRIOR ENGAGEMENT → RESPONSE TO ANOTHER MESSAGE")
print("=" * 130)

display(
    engagement_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 18. THIRD TABLE:
# PRIOR PAYMENT
# ============================================================

payment_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "had_prior_payment"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


payment_table["recovery_per_message"] = (
    payment_table["recovery_brl"]
    /
    payment_table["messages"]
)


print("\n" + "=" * 130)
print("PRIOR PAYMENT → RESPONSE TO ANOTHER MESSAGE")
print("=" * 130)

display(
    payment_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "avg_prior_msgs":
            "{:.1f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)

ONE MORE MESSAGE? — DPD 45-60


,dpd_stage,prior_msgs_bucket,recency_bucket,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,prior_engagement_rate,prior_payment_rate,avg_msgs_14d,recovery_per_message,wa_cost,observed_net,sample_flag
0,45-49,0-2,0-3d,1,1,0,0.00%,R$ 0.00,R$ 863.54,100.0%,0.0%,1.00,R$ 0.00,R$ 1.00,R$ -1.00,VERY_SMALL
1,45-49,0-2,4-7d,1,1,0,0.00%,R$ 0.00,R$ 503.40,0.0%,0.0%,1.00,R$ 0.00,R$ 1.00,R$ -1.00,VERY_SMALL
2,45-49,0-2,15+d,6,6,1,16.67%,"R$ 2,000.00","R$ 1,073.76",83.3%,16.7%,0.00,R$ 333.33,R$ 6.00,"R$ 1,994.00",VERY_SMALL
3,45-49,3-5,0-3d,16,14,1,6.25%,"R$ 2,000.00","R$ 1,026.91",87.5%,18.8%,1.62,R$ 125.00,R$ 16.00,"R$ 1,984.00",VERY_SMALL
4,45-49,3-5,4-7d,36,36,2,5.56%,R$ 509.46,R$ 795.84,83.3%,22.2%,1.39,R$ 14.15,R$ 36.00,R$ 473.46,SMALL
5,45-49,3-5,8-14d,52,52,6,11.54%,"R$ 1,983.43",R$ 674.75,88.5%,26.9%,1.04,R$ 38.14,R$ 52.00,"R$ 1,931.43",OK
6,45-49,3-5,15+d,91,91,10,10.99%,"R$ 6,868.14",R$ 803.02,85.7%,17.6%,0.03,R$ 75.47,R$ 91.00,"R$ 6,777.14",OK
7,45-49,6-8,0-3d,118,109,7,5.93%,"R$ 3,147.78",R$ 747.72,94.1%,17.8%,1.82,R$ 26.68,R$ 118.00,"R$ 3,029.78",GOOD
8,45-49,6-8,4-7d,180,178,11,6.11%,"R$ 5,679.97",R$ 790.09,88.3%,17.2%,1.57,R$ 31.56,R$ 180.00,"R$ 5,499.97",GOOD
9,45-49,6-8,8-14d,200,200,5,2.50%,"R$ 3,372.10",R$ 797.04,88.0%,13.5%,1.31,R$ 16.86,R$ 200.00,"R$ 3,172.10",GOOD



PRIOR ENGAGEMENT → RESPONSE TO ANOTHER MESSAGE


,dpd_stage,prior_engagement_profile,messages,customers,avg_prior_msgs,avg_days_since_prev_msg,payment_events,payment_rate,recovery_brl,avg_balance,recovery_per_message
0,45-49,no_prior_engagement,211,179,8.5,9.7,3,1.42%,"R$ 2,163.08",R$ 785.97,R$ 10.25
1,45-49,prior_click,982,832,9.0,8.6,46,4.68%,"R$ 27,685.33",R$ 773.82,R$ 28.19
2,45-49,prior_read_only,454,383,8.4,10.2,26,5.73%,"R$ 13,012.60",R$ 802.87,R$ 28.66
3,45-49,prior_reply,291,252,8.7,9.0,15,5.15%,"R$ 5,326.25",R$ 814.72,R$ 18.30
4,50-54,no_prior_engagement,187,152,9.2,9.7,0,0.00%,R$ 0.00,R$ 745.91,R$ 0.00
5,50-54,prior_click,991,819,9.3,9.3,66,6.66%,"R$ 33,509.27",R$ 779.27,R$ 33.81
6,50-54,prior_read_only,371,312,8.7,10.4,12,3.23%,"R$ 4,769.23",R$ 837.70,R$ 12.86
7,50-54,prior_reply,225,198,8.5,10.8,11,4.89%,"R$ 5,649.39",R$ 821.36,R$ 25.11
8,55-60,no_prior_engagement,207,162,9.3,10.6,1,0.48%,R$ 140.57,R$ 759.72,R$ 0.68
9,55-60,prior_click,932,756,9.9,9.6,44,4.72%,"R$ 23,997.92",R$ 780.53,R$ 25.75



PRIOR PAYMENT → RESPONSE TO ANOTHER MESSAGE


,dpd_stage,had_prior_payment,messages,customers,avg_prior_msgs,payment_events,payment_rate,recovery_brl,avg_balance,recovery_per_message
0,45-49,False,"1,632","1,375",8.8,64,3.92%,"R$ 41,982.84",R$ 866.93,R$ 25.72
1,45-49,True,306,259,8.6,26,8.50%,"R$ 6,204.42",R$ 367.57,R$ 20.28
2,50-54,False,"1,461","1,207",9.1,52,3.56%,"R$ 34,518.30",R$ 877.93,R$ 23.63
3,50-54,True,313,265,8.7,37,11.82%,"R$ 9,409.59",R$ 398.33,R$ 30.06
4,55-60,False,"1,397","1,125",9.7,58,4.15%,"R$ 40,312.80",R$ 852.68,R$ 28.86
5,55-60,True,272,226,9.2,23,8.46%,"R$ 6,954.66",R$ 394.93,R$ 25.57


In [7]:
# ============================================================
# SLIDE SUPPORT — LAST ATTEMPT BEFORE DPD 60
#
# Objective:
# Identify which customers still show historical evidence
# of response to ONE MORE WhatsApp contact in DPD 45-60.
#
# IMPORTANT:
# All segmentation variables are known BEFORE current message.
# Outcomes refer to the CURRENT message.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 0. BASE
# ============================================================

df = decision.copy()


# ------------------------------------------------------------
# Standardize prior engagement
# ------------------------------------------------------------

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)


# ------------------------------------------------------------
# Recent contact
#
# Definition:
# recent = previous WhatsApp <= 7 days ago
#
# We will ALSO keep exact recency bands so we can validate
# whether this threshold is reasonable.
# ------------------------------------------------------------

df["recent_contact_7d"] = (
    df["days_since_prev_msg"] <= 7
)

df["recent_contact_7d"] = (
    df["recent_contact_7d"]
    .fillna(False)
)


# ============================================================
# HELPER
# ============================================================

def summarize(group_cols):

    out = (
        df
        .groupby(
            group_cols,
            observed=True,
            dropna=False
        )
        .agg(
            messages=(
                "customer_id",
                "size"
            ),

            customers=(
                "customer_id",
                "nunique"
            ),

            payment_events=(
                "paid_72h",
                "sum"
            ),

            payment_rate=(
                "paid_72h",
                "mean"
            ),

            recovery_brl=(
                "recovery_72h",
                "sum"
            ),

            outstanding_brl=(
                "outstanding_balance_brl",
                "sum"
            ),

            avg_balance=(
                "outstanding_balance_brl",
                "mean"
            ),

            avg_prior_msgs=(
                "n_prior_msgs_total",
                "mean"
            ),

            avg_days_since_prev_msg=(
                "days_since_prev_msg",
                "mean"
            )
        )
        .reset_index()
    )

    out["recovery_per_message"] = (
        out["recovery_brl"]
        /
        out["messages"]
    )

    out["message_share"] = (
        out["messages"]
        /
        len(df)
    )

    return out


# ============================================================
# TABLE 1
# COMBINATION — ENGAGEMENT × PAYMENT
#
# This should be the FIRST evidence on the slide.
# ============================================================

combo = summarize([
    "has_prior_engagement",
    "has_prior_payment"
])


combo["profile"] = np.select(

    [
        combo["has_prior_engagement"]
        & combo["has_prior_payment"],

        combo["has_prior_engagement"]
        & ~combo["has_prior_payment"],

        ~combo["has_prior_engagement"]
        & combo["has_prior_payment"]
    ],

    [
        "Engaged + prior payment",
        "Engaged + no prior payment",
        "No engagement + prior payment"
    ],

    default="No engagement + no prior payment"
)


combo = (
    combo[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs",
            "avg_days_since_prev_msg",
            "message_share"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 120)
print("TABLE 1 — COMBINED WILLINGNESS SIGNAL")
print("ENGAGEMENT × PRIOR PAYMENT")
print("=" * 120)

display(
    combo.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}",
        "avg_days_since_prev_msg": "{:.1f}",
        "message_share": "{:.1%}"
    })
)


# ============================================================
# TABLE 2
# COMBINATION × DPD
#
# Critical:
# Does the combined signal survive until DPD 55-60?
# ============================================================

combo_dpd = summarize([
    "dpd_stage",
    "has_prior_engagement",
    "has_prior_payment"
])


combo_dpd["profile"] = np.select(

    [
        combo_dpd["has_prior_engagement"]
        & combo_dpd["has_prior_payment"],

        combo_dpd["has_prior_engagement"]
        & ~combo_dpd["has_prior_payment"],

        ~combo_dpd["has_prior_engagement"]
        & combo_dpd["has_prior_payment"]
    ],

    [
        "Engaged + prior payment",
        "Engaged + no prior payment",
        "No engagement + prior payment"
    ],

    default="No engagement + no prior payment"
)


combo_dpd = combo_dpd[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 2 — COMBINED SIGNAL × DPD")
print("=" * 120)

display(
    combo_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 3
# ENGAGEMENT ALONE
# ============================================================

engagement = summarize([
    "has_prior_engagement"
])


engagement["profile"] = np.where(
    engagement["has_prior_engagement"],
    "Prior engagement",
    "No prior engagement"
)


engagement = engagement[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 3 — PRIOR ENGAGEMENT")
print("=" * 120)

display(
    engagement.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 4
# PRIOR PAYMENT ALONE
# ============================================================

payment = summarize([
    "has_prior_payment"
])


payment["profile"] = np.where(
    payment["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)


payment = payment[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 4 — PRIOR PAYMENT")
print("=" * 120)

display(
    payment.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 5
# RECENCY ALONE
#
# First show granular recency.
# Do NOT jump directly to <=7 vs >7.
# ============================================================

recency = summarize([
    "recency_bucket"
])


recency = recency[
    [
        "recency_bucket",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance",
        "avg_prior_msgs"
    ]
]


print("\n" + "=" * 120)
print("TABLE 5 — DAYS SINCE PREVIOUS CONTACT")
print("=" * 120)

display(
    recency.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}"
    })
)


# ============================================================
# TABLE 6
# RECENT vs NOT RECENT
# ============================================================

recent = summarize([
    "recent_contact_7d"
])


recent["profile"] = np.where(
    recent["recent_contact_7d"],
    "Recent contact <=7d",
    "No recent contact >7d"
)


recent = recent[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance",
        "avg_prior_msgs"
    ]
]


print("\n" + "=" * 120)
print("TABLE 6 — RECENT CONTACT <=7 DAYS")
print("=" * 120)

display(
    recent.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}"
    })
)


# ============================================================
# TABLE 7
# ALL THREE SIGNALS TOGETHER
#
# Engagement × payment × recent contact
#
# This is the candidate RULE HIERARCHY table.
# ============================================================

all_signals = summarize([
    "has_prior_engagement",
    "has_prior_payment",
    "recent_contact_7d"
])


all_signals["profile"] = (
    np.where(
        all_signals["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        all_signals["has_prior_payment"],
        "Prior payment",
        "No payment"
    )
    + " | "
    +
    np.where(
        all_signals["recent_contact_7d"],
        "Contact <=7d",
        "Contact >7d"
    )
)


all_signals = (
    all_signals[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs",
            "avg_days_since_prev_msg"
        ]
    ]
    .sort_values(
        [
            "payment_rate",
            "recovery_per_message"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("TABLE 7 — ALL SIGNALS COMBINED")
print("ENGAGEMENT × PAYMENT × RECENCY")
print("=" * 120)

display(
    all_signals.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}",
        "avg_days_since_prev_msg": "{:.1f}"
    })
)


# ============================================================
# TABLE 8
# ALL SIGNALS × DPD
#
# This is our final robustness check before choosing timing.
# ============================================================

all_signals_dpd = summarize([
    "dpd_stage",
    "has_prior_engagement",
    "has_prior_payment",
    "recent_contact_7d"
])


all_signals_dpd["profile"] = (
    np.where(
        all_signals_dpd["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        all_signals_dpd["has_prior_payment"],
        "Prior payment",
        "No payment"
    )
    + " | "
    +
    np.where(
        all_signals_dpd["recent_contact_7d"],
        "Contact <=7d",
        "Contact >7d"
    )
)


all_signals_dpd = all_signals_dpd[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 8 — ALL SIGNALS × DPD")
print("=" * 120)

display(
    all_signals_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# QA / RECONCILIATION
# ============================================================

print("\n" + "=" * 120)
print("QA")
print("=" * 120)

print(f"Historical DPD45-60 messages : {len(df):,}")
print(f"Unique customers             : {df['customer_id'].nunique():,}")
print(f"Payment events               : {df['paid_72h'].sum():,}")
print(f"Recovery                     : R$ {df['recovery_72h'].sum():,.2f}")

print("\nCombination table messages:")
print(f"{combo['messages'].sum():,}")

print("\nAll-signals table messages:")
print(f"{all_signals['messages'].sum():,}")

assert combo["messages"].sum() == len(df)
assert all_signals["messages"].sum() == len(df)

print("\n✓ ALL HISTORICAL MESSAGES RECONCILED")

TABLE 1 — COMBINED WILLINGNESS SIGNAL
ENGAGEMENT × PRIOR PAYMENT


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs,avg_days_since_prev_msg,message_share
0,Engaged + prior payment,872,525,86,9.86%,R$ 25.88,"R$ 22,568.67",R$ 387.04,8.8,9.9,16.2%
1,Engaged + no prior payment,"3,904","2,183",170,4.35%,R$ 29.33,"R$ 114,510.29",R$ 879.40,9.2,9.7,72.6%
2,No engagement + no prior payment,586,328,4,0.68%,R$ 3.93,"R$ 2,303.65",R$ 777.32,9.0,10.0,10.9%
3,No engagement + prior payment,19,11,0,0.00%,R$ 0.00,R$ 0.00,R$ 372.64,7.2,10.8,0.4%



TABLE 2 — COMBINED SIGNAL × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance
0,45-49,No engagement + no prior payment,205,173,3,1.46%,R$ 10.55,R$ 799.42
1,45-49,No engagement + prior payment,6,6,0,0.00%,R$ 0.00,R$ 326.73
2,45-49,Engaged + no prior payment,"1,427","1,204",61,4.27%,R$ 27.90,R$ 876.63
3,45-49,Engaged + prior payment,300,253,26,8.67%,R$ 20.68,R$ 368.39
4,50-54,No engagement + no prior payment,182,148,0,0.00%,R$ 0.00,R$ 760.27
5,50-54,No engagement + prior payment,5,4,0,0.00%,R$ 0.00,R$ 223.29
6,50-54,Engaged + no prior payment,"1,279","1,059",52,4.07%,R$ 26.99,R$ 894.68
7,50-54,Engaged + prior payment,308,261,37,12.01%,R$ 30.55,R$ 401.17
8,55-60,No engagement + no prior payment,199,156,1,0.50%,R$ 0.71,R$ 770.14
9,55-60,No engagement + prior payment,8,6,0,0.00%,R$ 0.00,R$ 500.42



TABLE 3 — PRIOR ENGAGEMENT


...


TABLE 7 — ALL SIGNALS COMBINED
ENGAGEMENT × PAYMENT × RECENCY


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs,avg_days_since_prev_msg
0,Engaged | Prior payment | Contact >7d,454,414,52,11.45%,R$ 30.08,"R$ 13,654.95",R$ 366.72,8.1,15.6
1,Engaged | Prior payment | Contact <=7d,418,265,34,8.13%,R$ 21.32,"R$ 8,913.72",R$ 409.10,9.6,3.6
2,Engaged | No payment | Contact >7d,"2,013","1,804",91,4.52%,R$ 28.97,"R$ 58,313.14",R$ 870.10,8.6,15.5
3,Engaged | No payment | Contact <=7d,"1,891","1,202",79,4.18%,R$ 29.72,"R$ 56,197.15",R$ 889.30,9.8,3.5
4,No engagement | No payment | Contact <=7d,283,168,2,0.71%,R$ 1.25,R$ 353.13,R$ 763.90,9.8,3.5
5,No engagement | No payment | Contact >7d,303,272,2,0.66%,R$ 6.44,"R$ 1,950.52",R$ 789.85,8.4,16.1
6,No engagement | Prior payment | Contact >7d,9,8,0,0.00%,R$ 0.00,R$ 0.00,R$ 420.60,6.6,18.8
7,No engagement | Prior payment | Contact <=7d,10,8,0,0.00%,R$ 0.00,R$ 0.00,R$ 329.48,7.7,3.5



TABLE 8 — ALL SIGNALS × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance
0,45-49,No engagement | No payment | Contact >7d,107,107,2,1.87%,R$ 18.23,R$ 858.37
1,45-49,No engagement | No payment | Contact <=7d,98,81,1,1.02%,R$ 2.17,R$ 735.05
2,45-49,No engagement | Prior payment | Contact >7d,3,3,0,0.00%,R$ 0.00,R$ 356.26
3,45-49,No engagement | Prior payment | Contact <=7d,3,3,0,0.00%,R$ 0.00,R$ 297.19
4,45-49,Engaged | No payment | Contact >7d,739,739,34,4.60%,R$ 31.79,R$ 863.75
5,45-49,Engaged | No payment | Contact <=7d,688,580,27,3.92%,R$ 23.73,R$ 890.46
6,45-49,Engaged | Prior payment | Contact >7d,132,132,14,10.61%,R$ 26.52,R$ 354.62
7,45-49,Engaged | Prior payment | Contact <=7d,168,136,12,7.14%,R$ 16.09,R$ 379.22
8,50-54,No engagement | No payment | Contact >7d,94,94,0,0.00%,R$ 0.00,R$ 724.40
9,50-54,No engagement | No payment | Contact <=7d,88,71,0,0.00%,R$ 0.00,R$ 798.58



QA
Historical DPD45-60 messages : 5,381
Unique customers             : 3,019
Payment events               : 260
Recovery                     : R$ 139,382.61

Combination table messages:
5,381

All-signals table messages:
5,381

✓ ALL HISTORICAL MESSAGES RECONCILED


In [9]:
# ============================================================
# VALIDATION:
# PRIOR ENGAGEMENT × CONTACT PRESSURE / TIME WITHOUT STIMULUS
#
# Question:
# Does "no prior engagement" remain a bad segment even when
# the customer has NOT been stimulated recently?
#
# Grain:
# 1 row = historical send at DPD 45-60
# ============================================================

import pandas as pd
import numpy as np


df = decision.copy()


# ============================================================
# 1. STANDARDIZE VARIABLES
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["days_since_prev_msg"] = pd.to_numeric(
    df["days_since_prev_msg"],
    errors="coerce"
)


# ============================================================
# 2. CONTACT PRESSURE
#
# Most important split:
#
# 0 messages in last 14d
# versus
# >=1 message in last 14d
# ============================================================

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# More granular version

df["pressure_14d"] = pd.cut(
    df["n_msgs_last_14d"],
    bins=[
        -1,
        0,
        2,
        4,
        np.inf
    ],
    labels=[
        "0 msgs",
        "1-2 msgs",
        "3-4 msgs",
        "5+ msgs"
    ]
)


# ============================================================
# 3. HELPER
# ============================================================

def make_summary(data, group_cols):

    out = (
        data
        .groupby(
            group_cols,
            observed=True,
            dropna=False
        )
        .agg(

            messages=(
                "customer_id",
                "size"
            ),

            customers=(
                "customer_id",
                "nunique"
            ),

            payment_events=(
                "paid_72h",
                "sum"
            ),

            payment_rate=(
                "paid_72h",
                "mean"
            ),

            recovery_brl=(
                "recovery_72h",
                "sum"
            ),

            avg_balance=(
                "outstanding_balance_brl",
                "mean"
            ),

            avg_prior_msgs_total=(
                "n_prior_msgs_total",
                "mean"
            ),

            avg_msgs_last_14d=(
                "n_msgs_last_14d",
                "mean"
            ),

            avg_days_since_prev_msg=(
                "days_since_prev_msg",
                "mean"
            ),

            prior_payment_rate=(
                "has_prior_payment",
                "mean"
            )
        )
        .reset_index()
    )

    out["recovery_per_message"] = (
        out["recovery_brl"]
        /
        out["messages"]
    )

    return out


# ============================================================
# TABLE A
# ENGAGEMENT × ZERO CONTACTS LAST 14D
#
# THIS IS THE MAIN TABLE.
# ============================================================

table_a = make_summary(
    df,
    [
        "has_prior_engagement",
        "no_msg_last_14d"
    ]
)


table_a["profile"] = np.select(

    [
        table_a["has_prior_engagement"]
        & table_a["no_msg_last_14d"],

        table_a["has_prior_engagement"]
        & ~table_a["no_msg_last_14d"],

        ~table_a["has_prior_engagement"]
        & table_a["no_msg_last_14d"]
    ],

    [
        "Engaged | 0 msgs last 14d",
        "Engaged | >=1 msg last 14d",
        "No engagement | 0 msgs last 14d"
    ],

    default="No engagement | >=1 msg last 14d"
)


table_a = (
    table_a[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs_total",
            "avg_days_since_prev_msg",
            "prior_payment_rate"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 120)
print("TABLE A — ENGAGEMENT × CONTACT IN LAST 14 DAYS")
print("=" * 120)

display(
    table_a.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate": "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "recovery_brl":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}",

        "prior_payment_rate":
            "{:.1%}"
    })
)


# ============================================================
# TABLE B
# ENGAGEMENT × FULL PRESSURE BUCKET
#
# Shows deterioration as pressure increases.
# ============================================================

table_b = make_summary(
    df,
    [
        "has_prior_engagement",
        "pressure_14d"
    ]
)


table_b["engagement"] = np.where(
    table_b["has_prior_engagement"],
    "Prior engagement",
    "No prior engagement"
)


table_b = table_b[
    [
        "engagement",
        "pressure_14d",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance",
        "avg_prior_msgs_total"
    ]
]


print("\n" + "=" * 120)
print("TABLE B — ENGAGEMENT × CONTACT PRESSURE")
print("=" * 120)

display(
    table_b.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}"
    })
)


# ============================================================
# TABLE C
# ENGAGEMENT × 0 MSG LAST 14D × PRIOR PAYMENT
#
# This tests whether prior payment can "rescue" a customer
# without previous engagement.
# ============================================================

table_c = make_summary(
    df,
    [
        "has_prior_engagement",
        "has_prior_payment",
        "no_msg_last_14d"
    ]
)


table_c["profile"] = (
    np.where(
        table_c["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        table_c["has_prior_payment"],
        "Prior payment",
        "No prior payment"
    )
    + " | "
    +
    np.where(
        table_c["no_msg_last_14d"],
        "0 msgs 14d",
        ">=1 msg 14d"
    )
)


table_c = (
    table_c[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "avg_balance",
            "avg_prior_msgs_total",
            "avg_days_since_prev_msg"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("TABLE C — ENGAGEMENT × PAYMENT × 14D PRESSURE")
print("=" * 120)

display(
    table_c.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}"
    })
)


# ============================================================
# TABLE D
# MOST IMPORTANT ROBUSTNESS CHECK:
#
# ENGAGEMENT × ZERO MSG 14D × DPD
#
# Does the "reactivation" effect survive in 45-49,
# 50-54 and 55-60?
# ============================================================

table_d = make_summary(
    df,
    [
        "dpd_stage",
        "has_prior_engagement",
        "no_msg_last_14d"
    ]
)


table_d["profile"] = (
    np.where(
        table_d["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        table_d["no_msg_last_14d"],
        "0 msgs 14d",
        ">=1 msg 14d"
    )
)


table_d = table_d[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance",
        "avg_prior_msgs_total"
    ]
]


print("\n" + "=" * 120)
print("TABLE D — ENGAGEMENT × 14D PRESSURE × DPD")
print("=" * 120)

display(
    table_d.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}"
    })
)


# ============================================================
# QA
# ============================================================

print("\n" + "=" * 120)
print("QA")
print("=" * 120)

print(
    f"DPD45-60 messages : "
    f"{len(df):,}"
)

print(
    f"0 msgs last 14d   : "
    f"{df['no_msg_last_14d'].sum():,}"
)

print(
    f">=1 msg last 14d  : "
    f"{(~df['no_msg_last_14d']).sum():,}"
)

print(
    f"No engagement     : "
    f"{(~df['has_prior_engagement']).sum():,}"
)

print(
    f"Prior engagement  : "
    f"{df['has_prior_engagement'].sum():,}"
)

assert table_a["messages"].sum() == len(df)
assert table_b["messages"].sum() == len(df)
assert table_c["messages"].sum() == len(df)
assert table_d["messages"].sum() == len(df)

print("\n✓ ALL TABLES RECONCILED")

TABLE A — ENGAGEMENT × CONTACT IN LAST 14 DAYS


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs_total,avg_days_since_prev_msg,prior_payment_rate
0,Engaged | 0 msgs last 14d,"1,113","1,111",72,6.47%,R$ 32.71,"R$ 36,405.80",R$ 788.61,7.7,21.9,18.6%
1,Engaged | >=1 msg last 14d,"3,663","2,050",184,5.02%,R$ 27.48,"R$ 100,673.16",R$ 789.78,9.6,6.0,18.2%
2,No engagement | 0 msgs last 14d,150,150,2,1.33%,R$ 13.00,"R$ 1,950.52",R$ 760.31,7.4,22.4,4.0%
3,No engagement | >=1 msg last 14d,455,247,2,0.44%,R$ 0.78,R$ 353.13,R$ 766.03,9.5,5.9,2.9%



TABLE B — ENGAGEMENT × CONTACT PRESSURE


,engagement,pressure_14d,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total
0,No prior engagement,0 msgs,150,150,2,1.33%,R$ 13.00,R$ 760.31,7.4
1,No prior engagement,1-2 msgs,354,230,2,0.56%,R$ 1.00,R$ 771.90,9.1
2,No prior engagement,3-4 msgs,98,66,0,0.00%,R$ 0.00,R$ 741.00,11.0
3,No prior engagement,5+ msgs,3,2,0,0.00%,R$ 0.00,R$ 890.17,14.3
4,Prior engagement,0 msgs,"1,113","1,111",72,6.47%,R$ 32.71,R$ 788.61,7.7
5,Prior engagement,1-2 msgs,"2,992","1,920",160,5.35%,R$ 28.54,R$ 789.21,9.2
6,Prior engagement,3-4 msgs,635,442,22,3.46%,R$ 22.55,R$ 794.94,10.9
7,Prior engagement,5+ msgs,36,26,2,5.56%,R$ 26.53,R$ 745.57,14.2



TABLE C — ENGAGEMENT × PAYMENT × 14D PRESSURE


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total,avg_days_since_prev_msg
0,Engaged | Prior payment | 0 msgs 14d,207,207,27,13.04%,R$ 35.84,R$ 362.23,7.4,22.0
1,Engaged | Prior payment | >=1 msg 14d,665,392,59,8.87%,R$ 22.78,R$ 394.76,9.3,6.1
2,Engaged | No prior payment | 0 msgs 14d,906,904,45,4.97%,R$ 31.99,R$ 886.03,7.8,21.9
3,Engaged | No prior payment | >=1 msg 14d,"2,998","1,671",125,4.17%,R$ 28.53,R$ 877.40,9.6,6.0
4,No engagement | No prior payment | 0 msgs 14d,144,144,2,1.39%,R$ 13.55,R$ 775.04,7.4,22.4
5,No engagement | No prior payment | >=1 msg 14d,442,239,2,0.45%,R$ 0.80,R$ 778.06,9.6,6.0
6,No engagement | Prior payment | 0 msgs 14d,6,6,0,0.00%,R$ 0.00,R$ 406.81,6.7,23.9
7,No engagement | Prior payment | >=1 msg 14d,13,8,0,0.00%,R$ 0.00,R$ 356.87,7.4,4.8



TABLE D — ENGAGEMENT × 14D PRESSURE × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total
0,45-49,No engagement | >=1 msg 14d,158,132,1,0.63%,R$ 1.35,R$ 775.54,9.0
1,45-49,No engagement | 0 msgs 14d,53,53,2,3.77%,R$ 36.80,R$ 817.07,7.1
2,45-49,Engaged | >=1 msg 14d,"1,356","1,130",60,4.42%,R$ 20.74,R$ 785.51,9.2
3,45-49,Engaged | 0 msgs 14d,371,371,27,7.28%,R$ 48.27,R$ 798.69,7.3
4,50-54,No engagement | >=1 msg 14d,147,119,0,0.00%,R$ 0.00,R$ 760.65,9.8
5,50-54,No engagement | 0 msgs 14d,40,40,0,0.00%,R$ 0.00,R$ 691.75,7.0
6,50-54,Engaged | >=1 msg 14d,"1,219","1,009",64,5.25%,R$ 27.83,R$ 790.14,9.5
7,50-54,Engaged | 0 msgs 14d,368,368,25,6.79%,R$ 27.18,R$ 827.91,7.5
8,55-60,No engagement | >=1 msg 14d,150,117,1,0.67%,R$ 0.94,R$ 761.27,9.9
9,55-60,No engagement | 0 msgs 14d,57,57,0,0.00%,R$ 0.00,R$ 755.64,7.9



QA
DPD45-60 messages : 5,381
0 msgs last 14d   : 1,263
>=1 msg last 14d  : 4,118
No engagement     : 605
Prior engagement  : 4,776

✓ ALL TABLES RECONCILED


In [10]:
# ============================================================
# FINAL RULE TABLE — ONE MORE MESSAGE DPD45-60
#
# Mutually exclusive rules
# Comparison against historical DPD45-60 baseline
#
# IMPORTANT:
# "uplift" below is OBSERVED association, NOT causal incrementality.
# ============================================================

import pandas as pd
import numpy as np

df = decision.copy()


# ============================================================
# 1. STANDARDIZE SIGNALS
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. BASELINE — ALL DPD45-60 MESSAGES
# ============================================================

BASELINE_MESSAGES = len(df)

BASELINE_PAYMENT_RATE = (
    df["paid_72h"].mean()
)

BASELINE_RECOVERY = (
    df["recovery_72h"].sum()
)

BASELINE_RECOVERY_PER_MSG = (
    BASELINE_RECOVERY
    / BASELINE_MESSAGES
)

BASELINE_COST_PER_MSG = 1.0

BASELINE_NET_PER_MSG = (
    BASELINE_RECOVERY_PER_MSG
    - BASELINE_COST_PER_MSG
)


print("=" * 100)
print("HISTORICAL BASELINE — DPD45-60")
print("=" * 100)

print(
    f"Messages              : "
    f"{BASELINE_MESSAGES:,}"
)

print(
    f"Payment rate          : "
    f"{BASELINE_PAYMENT_RATE:.2%}"
)

print(
    f"Recovery / message    : "
    f"R$ {BASELINE_RECOVERY_PER_MSG:,.2f}"
)

print(
    f"Net recovery / message: "
    f"R$ {BASELINE_NET_PER_MSG:,.2f}"
)


# ============================================================
# 3. MUTUALLY EXCLUSIVE RULES
# ============================================================

conditions = [

    # R1
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R2
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R3
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R4
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R5
    (
        ~df["has_prior_engagement"]
        & df["no_msg_last_14d"]
    ),

    # R6
    (
        ~df["has_prior_engagement"]
        & ~df["no_msg_last_14d"]
    )
]


labels = [

    "R1 | Engaged + prior payment + 0 msgs/14d",

    "R2 | Engaged + prior payment + recent contact",

    "R3 | Engaged + no prior payment + 0 msgs/14d",

    "R4 | Engaged + no prior payment + recent contact",

    "R5 | No engagement + 0 msgs/14d",

    "R6 | No engagement + recent contact"
]


df["rule"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED"
)


# ============================================================
# 4. AGGREGATE
# ============================================================

rule_table = (
    df
    .groupby(
        "rule",
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. ECONOMICS
# ============================================================

rule_table["recovery_per_message"] = (
    rule_table["recovery_brl"]
    /
    rule_table["messages"]
)


# WhatsApp = R$1 per attempt

rule_table["cost_brl"] = (
    rule_table["messages"]
    * 1.0
)


rule_table["net_recovery_brl"] = (
    rule_table["recovery_brl"]
    -
    rule_table["cost_brl"]
)


rule_table["net_per_message"] = (
    rule_table["recovery_per_message"]
    - 1
)


# ============================================================
# 6. COMPARISON AGAINST BASELINE
# ============================================================

rule_table["delta_payment_rate_pp"] = (
    (
        rule_table["payment_rate"]
        - BASELINE_PAYMENT_RATE
    )
    * 100
)


rule_table["delta_recovery_per_msg"] = (
    rule_table["recovery_per_message"]
    - BASELINE_RECOVERY_PER_MSG
)


rule_table["recovery_index"] = (
    rule_table["recovery_per_message"]
    /
    BASELINE_RECOVERY_PER_MSG
    * 100
)


# Example:
# 138 = R$/msg is 38% above historical baseline


rule_table["recovery_uplift_pct"] = (
    (
        rule_table["recovery_per_message"]
        /
        BASELINE_RECOVERY_PER_MSG
    )
    - 1
)


# ============================================================
# 7. ORDER BY BUSINESS RULE
# ============================================================

rule_order = {
    label: i + 1
    for i, label in enumerate(labels)
}

rule_table["priority"] = (
    rule_table["rule"]
    .map(rule_order)
)


rule_table = (
    rule_table
    .sort_values("priority")
    .reset_index(drop=True)
)


# ============================================================
# 8. BUSINESS DECISION LABEL
#
# Deliberately descriptive.
# We will validate thresholds from the output.
# ============================================================

rule_table["candidate_action"] = np.select(

    [
        rule_table["priority"].isin([1, 2, 3]),

        rule_table["priority"].eq(4),

        rule_table["priority"].eq(5),

        rule_table["priority"].eq(6)
    ],

    [
        "PRIORITIZE",
        "CONTACT / LOWER PRIORITY",
        "TEST / REACTIVATION",
        "SUPPRESS"
    ],

    default="REVIEW"
)


# ============================================================
# 9. FINAL SLIDE TABLE
# ============================================================

slide_table = rule_table[
    [
        "priority",
        "rule",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "delta_recovery_per_msg",
        "recovery_uplift_pct",
        "recovery_index",
        "net_per_message",
        "avg_balance",
        "candidate_action"
    ]
].copy()


print("\n" + "=" * 130)
print("FINAL RULE TABLE — HISTORICAL PERFORMANCE")
print("=" * 130)

display(
    slide_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "delta_recovery_per_msg":
            "R$ {:+,.2f}",

        "recovery_uplift_pct":
            "{:+.1%}",

        "recovery_index":
            "{:.0f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 10. QA
# ============================================================

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(
    f"Total historical messages : "
    f"{len(df):,}"
)

print(
    f"Messages classified       : "
    f"{rule_table['messages'].sum():,}"
)

print(
    f"Unclassified              : "
    f"{(df['rule'] == 'UNCLASSIFIED').sum():,}"
)

print(
    f"\nBaseline payment rate     : "
    f"{BASELINE_PAYMENT_RATE:.2%}"
)

print(
    f"Baseline recovery/msg     : "
    f"R$ {BASELINE_RECOVERY_PER_MSG:,.2f}"
)

assert (
    rule_table["messages"].sum()
    == len(df)
)

assert (
    (df["rule"] == "UNCLASSIFIED").sum()
    == 0
)

print("\n✓ ALL MESSAGES CLASSIFIED")

HISTORICAL BASELINE — DPD45-60
Messages              : 5,381
Payment rate          : 4.83%
Recovery / message    : R$ 25.90
Net recovery / message: R$ 24.90

FINAL RULE TABLE — HISTORICAL PERFORMANCE


,priority,rule,messages,customers,payment_events,payment_rate,recovery_per_message,delta_recovery_per_msg,recovery_uplift_pct,recovery_index,net_per_message,avg_balance,candidate_action
0,1,R1 | Engaged + prior payment + 0 msgs/14d,207,207,27,13.04%,R$ 35.84,R$ +9.94,+38.4%,138,R$ 34.84,R$ 362.23,PRIORITIZE
1,2,R2 | Engaged + prior payment + recent contact,665,392,59,8.87%,R$ 22.78,R$ -3.12,-12.0%,88,R$ 21.78,R$ 394.76,PRIORITIZE
2,3,R3 | Engaged + no prior payment + 0 msgs/14d,906,904,45,4.97%,R$ 31.99,R$ +6.09,+23.5%,124,R$ 30.99,R$ 886.03,PRIORITIZE
3,4,R4 | Engaged + no prior payment + recent contact,"2,998","1,671",125,4.17%,R$ 28.53,R$ +2.62,+10.1%,110,R$ 27.53,R$ 877.40,CONTACT / LOWER PRIORITY
4,5,R5 | No engagement + 0 msgs/14d,150,150,2,1.33%,R$ 13.00,R$ -12.90,-49.8%,50,R$ 12.00,R$ 760.31,TEST / REACTIVATION
5,6,R6 | No engagement + recent contact,455,247,2,0.44%,R$ 0.78,R$ -25.13,-97.0%,3,R$ -0.22,R$ 766.03,SUPPRESS



QA
Total historical messages : 5,381
Messages classified       : 5,381
Unclassified              : 0

Baseline payment rate     : 4.83%
Baseline recovery/msg     : R$ 25.90

✓ ALL MESSAGES CLASSIFIED


In [11]:
# ============================================================
# FINAL POLICY VALIDATION
# SELECTED RULES (R1 + R3 + R4) vs REST
#
# Goal:
# Validate whether the proposed policy concentrates
# higher observed recovery per message.
#
# IMPORTANT:
# Historical association — NOT causal incrementality.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. BASE
# ============================================================

df = decision.copy()


# ------------------------------------------------------------
# Standardize signals
# ------------------------------------------------------------

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. RECREATE THE SIX MUTUALLY EXCLUSIVE RULES
# ============================================================

conditions = [

    # R1
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R2
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R3
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R4
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R5
    (
        ~df["has_prior_engagement"]
        & df["no_msg_last_14d"]
    ),

    # R6
    (
        ~df["has_prior_engagement"]
        & ~df["no_msg_last_14d"]
    )
]


labels = [
    "R1",
    "R2",
    "R3",
    "R4",
    "R5",
    "R6"
]


df["rule"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED"
)


# ============================================================
# 3. FINAL POLICY
#
# Selected:
# R1 + R3 + R4
#
# Rest:
# R2 + R5 + R6
# ============================================================

selected_rules = ["R1", "R3", "R4"]

df["policy_group"] = np.where(
    df["rule"].isin(selected_rules),
    "SEND — R1 + R3 + R4",
    "REST — R2 + R5 + R6"
)


# ============================================================
# 4. AGGREGATE
# ============================================================

policy_table = (
    df
    .groupby(
        "policy_group",
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. ECONOMICS
# ============================================================

policy_table["recovery_per_message"] = (
    policy_table["recovery_brl"]
    /
    policy_table["messages"]
)


policy_table["cost_brl"] = (
    policy_table["messages"]
    * 1
)


policy_table["net_recovery_brl"] = (
    policy_table["recovery_brl"]
    -
    policy_table["cost_brl"]
)


policy_table["net_per_message"] = (
    policy_table["recovery_per_message"]
    - 1
)


# ============================================================
# 6. POPULATION / RECOVERY SHARES
# ============================================================

policy_table["message_share"] = (
    policy_table["messages"]
    /
    policy_table["messages"].sum()
)


policy_table["recovery_share"] = (
    policy_table["recovery_brl"]
    /
    policy_table["recovery_brl"].sum()
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

policy_table = policy_table[
    [
        "policy_group",
        "messages",
        "customers",
        "message_share",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_share",
        "recovery_per_message",
        "net_per_message",
        "avg_balance",
        "avg_prior_msgs",
        "avg_msgs_last_14d"
    ]
]


print("=" * 130)
print("FINAL POLICY — SELECTED RULES vs REST")
print("=" * 130)

display(
    policy_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "message_share":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_share":
            "{:.1%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_msgs_last_14d":
            "{:.1f}"
    })
)


# ============================================================
# 8. DIRECT COMPARISON
# ============================================================

send = policy_table.loc[
    policy_table["policy_group"]
    == "SEND — R1 + R3 + R4"
].iloc[0]

rest = policy_table.loc[
    policy_table["policy_group"]
    == "REST — R2 + R5 + R6"
].iloc[0]


delta_recovery_msg = (
    send["recovery_per_message"]
    -
    rest["recovery_per_message"]
)


relative_recovery_msg = (
    send["recovery_per_message"]
    /
    rest["recovery_per_message"]
    - 1
)


delta_payment_rate_pp = (
    send["payment_rate"]
    -
    rest["payment_rate"]
) * 100


print("\n" + "=" * 130)
print("SELECTED POLICY vs REST")
print("=" * 130)

print(
    f"Recovery/msg — selected : "
    f"R$ {send['recovery_per_message']:,.2f}"
)

print(
    f"Recovery/msg — rest     : "
    f"R$ {rest['recovery_per_message']:,.2f}"
)

print(
    f"Difference              : "
    f"R$ {delta_recovery_msg:+,.2f} / msg"
)

print(
    f"Relative difference     : "
    f"{relative_recovery_msg:+.1%}"
)

print(
    f"\nPayment rate — selected : "
    f"{send['payment_rate']:.2%}"
)

print(
    f"Payment rate — rest     : "
    f"{rest['payment_rate']:.2%}"
)

print(
    f"Difference              : "
    f"{delta_payment_rate_pp:+.2f} pp"
)


# ============================================================
# 9. QA
# ============================================================

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(
    f"Historical messages : "
    f"{len(df):,}"
)

print(
    f"Table messages      : "
    f"{policy_table['messages'].sum():,}"
)

print(
    f"Unclassified        : "
    f"{(df['rule'] == 'UNCLASSIFIED').sum():,}"
)

assert (
    policy_table["messages"].sum()
    == len(df)
)

assert (
    (df["rule"] == "UNCLASSIFIED").sum()
    == 0
)

print("\n✓ RECONCILED")

FINAL POLICY — SELECTED RULES vs REST


,policy_group,messages,customers,message_share,payment_events,payment_rate,recovery_brl,recovery_share,recovery_per_message,net_per_message,avg_balance,avg_prior_msgs,avg_msgs_last_14d
0,REST — R2 + R5 + R6,"1,270",730,23.6%,63,4.96%,"R$ 17,453.52",12.5%,R$ 13.74,R$ 12.74,R$ 570.95,9.1,1.6
1,SEND — R1 + R3 + R4,"4,111","2,390",76.4%,197,4.79%,"R$ 121,929.09",87.5%,R$ 29.66,R$ 28.66,R$ 853.36,9.1,1.3



SELECTED POLICY vs REST
Recovery/msg — selected : R$ 29.66
Recovery/msg — rest     : R$ 13.74
Difference              : R$ +15.92 / msg
Relative difference     : +115.8%

Payment rate — selected : 4.79%
Payment rate — rest     : 4.96%
Difference              : -0.17 pp

QA
Historical messages : 5,381
Table messages      : 5,381
Unclassified        : 0

✓ RECONCILED


In [12]:
# ============================================================
# FINAL OPEN TABLE
#
# TOTAL UNIVERSE
#   ├── R1
#   ├── R2
#   ├── R3
#   └── REST
#
# R1/R2/R3 = selected rules
# REST      = everything else
# ============================================================

import pandas as pd
import numpy as np


df = decision.copy()


# ============================================================
# 1. STANDARDIZE SIGNALS
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. FINAL SELECTED RULES
# ============================================================

r1 = (
    df["has_prior_engagement"]
    & df["has_prior_payment"]
    & df["no_msg_last_14d"]
)

r2 = (
    df["has_prior_engagement"]
    & ~df["has_prior_payment"]
    & df["no_msg_last_14d"]
)

r3 = (
    df["has_prior_engagement"]
    & ~df["has_prior_payment"]
    & ~df["no_msg_last_14d"]
)


df["final_rule"] = np.select(
    [r1, r2, r3],
    [
        "R1 | Engaged + prior payment + 0 msgs/14d",
        "R2 | Engaged + no prior payment + 0 msgs/14d",
        "R3 | Engaged + no prior payment + recent contact"
    ],
    default="REST"
)


# ============================================================
# 3. SUMMARY FUNCTION
# ============================================================

def calculate_metrics(data):

    messages = len(data)

    customers = data["customer_id"].nunique()

    payment_events = data["paid_72h"].sum()

    payment_rate = (
        data["paid_72h"].mean()
        if messages > 0
        else np.nan
    )

    recovery = data["recovery_72h"].sum()

    recovery_per_message = (
        recovery / messages
        if messages > 0
        else np.nan
    )

    avg_balance = (
        data["outstanding_balance_brl"].mean()
    )

    return {
        "messages": messages,
        "customers": customers,
        "payment_events": payment_events,
        "payment_rate": payment_rate,
        "recovery_brl": recovery,
        "recovery_per_message": recovery_per_message,
        "avg_balance": avg_balance
    }


# ============================================================
# 4. TOTAL UNIVERSE
# ============================================================

rows = []

total_metrics = calculate_metrics(df)

rows.append({
    "segment": "TOTAL | DPD45-60",
    **total_metrics
})


# ============================================================
# 5. EACH SELECTED RULE + REST
# ============================================================

rule_order = [
    "R1 | Engaged + prior payment + 0 msgs/14d",
    "R2 | Engaged + no prior payment + 0 msgs/14d",
    "R3 | Engaged + no prior payment + recent contact",
    "REST"
]


for rule in rule_order:

    subset = df.loc[
        df["final_rule"].eq(rule)
    ]

    metrics = calculate_metrics(subset)

    rows.append({
        "segment": rule,
        **metrics
    })


# ============================================================
# 6. FINAL TABLE
# ============================================================

final_table = pd.DataFrame(rows)


# ------------------------------------------------------------
# Shares relative to TOTAL universe
# ------------------------------------------------------------

total_messages = len(df)
total_recovery = df["recovery_72h"].sum()


final_table["message_share"] = (
    final_table["messages"]
    / total_messages
)


final_table["recovery_share"] = (
    final_table["recovery_brl"]
    / total_recovery
)


# ------------------------------------------------------------
# Difference vs TOTAL historical average
# ------------------------------------------------------------

baseline_rpm = (
    total_recovery
    / total_messages
)


final_table["delta_rpm_vs_total"] = (
    final_table["recovery_per_message"]
    - baseline_rpm
)


# ============================================================
# 7. DISPLAY
# ============================================================

final_table = final_table[
    [
        "segment",
        "messages",
        "customers",
        "message_share",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_share",
        "recovery_per_message",
        "delta_rpm_vs_total",
        "avg_balance"
    ]
]


print("=" * 140)
print("DPD45-60 — FINAL RULE DECOMPOSITION")
print("=" * 140)

display(
    final_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "message_share":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_share":
            "{:.1%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "delta_rpm_vs_total":
            "R$ {:+,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 8. RECONCILIATION
#
# R1 + R2 + R3 + REST must equal TOTAL
# ============================================================

detail = final_table.iloc[1:]

print("\n" + "=" * 100)
print("RECONCILIATION")
print("=" * 100)

print(
    f"TOTAL messages       : "
    f"{total_messages:,}"
)

print(
    f"R1 + R2 + R3 + REST : "
    f"{detail['messages'].sum():,.0f}"
)

print(
    f"\nTOTAL recovery       : "
    f"R$ {total_recovery:,.2f}"
)

print(
    f"Rules + REST recovery: "
    f"R$ {detail['recovery_brl'].sum():,.2f}"
)

assert (
    detail["messages"].sum()
    == total_messages
)

assert np.isclose(
    detail["recovery_brl"].sum(),
    total_recovery
)

print("\n✓ R1 + R2 + R3 + REST = TOTAL")

DPD45-60 — FINAL RULE DECOMPOSITION


,segment,messages,customers,message_share,payment_events,payment_rate,recovery_brl,recovery_share,recovery_per_message,delta_rpm_vs_total,avg_balance
0,TOTAL | DPD45-60,"5,381","3,019",100.0%,260,4.83%,"R$ 139,382.61",100.0%,R$ 25.90,R$ +0.00,R$ 786.71
1,R1 | Engaged + prior payment + 0 msgs/14d,207,207,3.8%,27,13.04%,"R$ 7,418.80",5.3%,R$ 35.84,R$ +9.94,R$ 362.23
2,R2 | Engaged + no prior payment + 0 msgs/14d,906,904,16.8%,45,4.97%,"R$ 28,987.00",20.8%,R$ 31.99,R$ +6.09,R$ 886.03
3,R3 | Engaged + no prior payment + recent contact,"2,998","1,671",55.7%,125,4.17%,"R$ 85,523.29",61.4%,R$ 28.53,R$ +2.62,R$ 877.40
4,REST,"1,270",730,23.6%,63,4.96%,"R$ 17,453.52",12.5%,R$ 13.74,R$ -12.16,R$ 570.95



RECONCILIATION
TOTAL messages       : 5,381
R1 + R2 + R3 + REST : 5,381

TOTAL recovery       : R$ 139,382.61
Rules + REST recovery: R$ 139,382.61

✓ R1 + R2 + R3 + REST = TOTAL


In [13]:
# ============================================================
# TEMPLATE PERFORMANCE WITHIN SELECTED RULES
#
# Population:
# DPD45-60
# Only selected profiles R1 / R2 / R3
#
# Question:
# Within comparable selected profiles, which template is
# historically associated with higher payment / recovery?
#
# IMPORTANT:
# Observational association — NOT causal template effect.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. SELECT ONLY R1 / R2 / R3
# ============================================================

selected = df.loc[
    df["final_rule"].ne("REST")
].copy()


print("=" * 110)
print("SELECTED POPULATION")
print("=" * 110)

print(f"Messages  : {len(selected):,}")
print(f"Customers : {selected['customer_id'].nunique():,}")

print("\nTemplates:")
print(
    selected["template"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# 2. RULE × TEMPLATE
# ============================================================

template_table = (
    selected
    .groupby(
        ["final_rule", "template"],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_dpd=(
            "days_past_due",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 3. ECONOMICS
# ============================================================

template_table["recovery_per_message"] = (
    template_table["recovery_brl"]
    /
    template_table["messages"]
)

template_table["net_per_message"] = (
    template_table["recovery_per_message"]
    - 1
)


# ============================================================
# 4. SHARE OF EACH TEMPLATE WITHIN RULE
#
# Important for common-support check:
# Was each template actually used enough inside each profile?
# ============================================================

rule_totals = (
    template_table
    .groupby("final_rule")["messages"]
    .transform("sum")
)

template_table["template_share_within_rule"] = (
    template_table["messages"]
    /
    rule_totals
)


# ============================================================
# 5. SAMPLE SIZE FLAG
# ============================================================

template_table["sample_flag"] = np.select(
    [
        template_table["messages"] < 30,
        template_table["messages"] < 100
    ],
    [
        "VERY SMALL",
        "SMALL"
    ],
    default="OK"
)


# ============================================================
# 6. ORDER
# ============================================================

rule_order = {
    "R1 | Engaged + prior payment + 0 msgs/14d": 1,
    "R2 | Engaged + no prior payment + 0 msgs/14d": 2,
    "R3 | Engaged + no prior payment + recent contact": 3
}

template_table["rule_order"] = (
    template_table["final_rule"]
    .map(rule_order)
)

template_table = (
    template_table
    .sort_values(
        [
            "rule_order",
            "recovery_per_message"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

result = template_table[
    [
        "final_rule",
        "template",
        "messages",
        "customers",
        "template_share_within_rule",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_per_message",
        "net_per_message",
        "avg_balance",
        "avg_dpd",
        "avg_prior_msgs",
        "avg_msgs_last_14d",
        "sample_flag"
    ]
].copy()


display(
    result.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "template_share_within_rule":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_dpd":
            "{:.1f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_msgs_last_14d":
            "{:.1f}"
    })
)


# ============================================================
# 8. QA
# ============================================================

print("\n" + "=" * 110)
print("QA")
print("=" * 110)

print(
    f"Selected messages       : "
    f"{len(selected):,}"
)

print(
    f"Table messages          : "
    f"{template_table['messages'].sum():,}"
)

print(
    f"Selected recovery       : "
    f"R$ {selected['recovery_72h'].sum():,.2f}"
)

print(
    f"Table recovery          : "
    f"R$ {template_table['recovery_brl'].sum():,.2f}"
)

assert template_table["messages"].sum() == len(selected)

assert np.isclose(
    template_table["recovery_brl"].sum(),
    selected["recovery_72h"].sum()
)

print("\n✓ RECONCILED")

SELECTED POPULATION
Messages  : 4,111
Customers : 2,390

Templates:
template
discount_offer     2047
urgent_reminder    1197
pix_link            867


,final_rule,template,messages,customers,template_share_within_rule,payment_events,payment_rate,recovery_brl,recovery_per_message,net_per_message,avg_balance,avg_dpd,avg_prior_msgs,avg_msgs_last_14d,sample_flag
0,R1 | Engaged + prior payment + 0 msgs/14d,discount_offer,95,95,45.9%,16,16.84%,"R$ 4,414.42",R$ 46.47,R$ 45.47,R$ 358.97,52.9,7.3,0.0,SMALL
1,R1 | Engaged + prior payment + 0 msgs/14d,urgent_reminder,72,72,34.8%,7,9.72%,"R$ 1,987.54",R$ 27.60,R$ 26.60,R$ 353.23,51.4,7.4,0.0,SMALL
2,R1 | Engaged + prior payment + 0 msgs/14d,pix_link,40,40,19.3%,4,10.00%,"R$ 1,016.84",R$ 25.42,R$ 24.42,R$ 386.18,53.7,7.6,0.0,SMALL
3,R2 | Engaged + no prior payment + 0 msgs/14d,discount_offer,446,446,49.2%,28,6.28%,"R$ 17,168.49",R$ 38.49,R$ 37.49,R$ 851.03,51.8,7.8,0.0,OK
4,R2 | Engaged + no prior payment + 0 msgs/14d,pix_link,198,198,21.9%,10,5.05%,"R$ 5,435.57",R$ 27.45,R$ 26.45,R$ 902.51,52.3,8.0,0.0,OK
5,R2 | Engaged + no prior payment + 0 msgs/14d,urgent_reminder,262,260,28.9%,7,2.67%,"R$ 6,382.94",R$ 24.36,R$ 23.36,R$ 933.14,51.9,7.8,0.0,OK
6,R3 | Engaged + no prior payment + recent contact,urgent_reminder,863,716,28.8%,35,4.06%,"R$ 29,105.66",R$ 33.73,R$ 32.73,R$ 883.78,51.6,9.7,1.8,OK
7,R3 | Engaged + no prior payment + recent contact,pix_link,629,557,21.0%,21,3.34%,"R$ 17,192.62",R$ 27.33,R$ 26.33,R$ 883.37,51.7,9.5,1.7,OK
8,R3 | Engaged + no prior payment + recent contact,discount_offer,"1,506","1,099",50.2%,69,4.58%,"R$ 39,225.01",R$ 26.05,R$ 25.05,R$ 871.24,51.7,9.6,1.8,OK



QA
Selected messages       : 4,111
Table messages          : 4,111
Selected recovery       : R$ 121,929.09
Table recovery          : R$ 121,929.09

✓ RECONCILED
